# Exercise 0 - PydanticAI and OpenRouter

3. A grading assistant

a) Go into this page with examples of students answers to a particular question. Copy some example texts and paste it into files with names like student_text_1.txt, student_text_2.txt.

b) Read these data into python and tell your LLM to grade them.

Läs in data + kalla LLM

Mål: loopa igenom texter och få bedömning.

In [1]:
from pathlib import Path

def load_texts(folder="data"):
    texts = []
    for file in Path(folder).glob("student_text_*.txt"):
        texts.append(file.read_text(encoding="utf-8"))
    return texts

texts = load_texts()

c) Make structured output with fields proposed_grade, motivation and improvements.

In [3]:
from pydantic import BaseModel
from typing import Literal

class GradeOutput(BaseModel):
    proposed_grade: Literal["A", "B", "C", "D", "E", "F"]
    motivation: str
    improvements: str

In [5]:
from pydantic_ai import Agent

agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    output_type=GradeOutput,
    system_prompt="""
Du är en erfaren svensklärare.
Bedöm elevtexten och ge:
- proposed_grade (A-F)
- motivation (kort motivering)
- improvements (konkreta förbättringar)
"""
)

Kör inferens

In [6]:
results = []

for text in texts:
    result = agent.run_sync(text)
    results.append(result)

d) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

Mål: separera output i tre filer.

In [10]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

for i, r in enumerate(results):
    (output_dir / f"student_{i}_grade.txt").write_text(r.proposed_grade)
    (output_dir / f"student_{i}_motivation.txt").write_text(r.motivation)
    (output_dir / f"student_{i}_improvements.txt").write_text(r.improvements)

e) Go into skolverket for Svenska 1 and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Paste this into a file called criterias.txt.

f) Now repeat b)-e) but with the criterias in your prompt as well. Can you see any differences in the outputs?

g) Can you improve the output quality by providing few shot examples?